# NB26 — FastF1 Pretrained Hybrid GRU+FC (Fine-Tune on Competition)

**Goal:** Fine-tune the HybridGRUFC model that was pretrained on real F1 lap data (2022–2024, ~150k laps from 66+ races) to the competition's synthetic dataset. The pretrained weights encode actual tyre physics before fine-tuning adapts them to the perturbed competition distribution.

## Rationale

The gap to 1st place (0.018) likely reflects that top competitors have better representations of the underlying tyre pit-stop dynamics. The competition data is synthetic/perturbed from real F1 data — pretraining on the real source teaches:
- Actual tyre degradation curves and cliff thresholds per compound
- Driver-specific stint length strategies
- Circuit-specific pit window timing patterns
- The GRU hidden state learns to encode 'about to cliff' tyre trajectories before seeing the competition data

## Prerequisites for Kaggle

Before running this notebook, upload these to Kaggle as a dataset:
- `models/hybrid_pretrained.pt` — pretrained model weights (from `scripts/pretrain_hybrid.py`)
- `models/hybrid_pretrain_meta.pkl` — scalers + label encoders from pretraining

**IMPORTANT:** The competition's label encoders (Driver/Race/Compound/Year) are re-fitted on combined train+test in this notebook. They may differ from the pretraining encoders (real F1 had different drivers/races). We handle this by re-initializing the embedding layers with fresh random weights and keeping only the GRU and FC branch weights from pretraining.

## Fine-Tuning Strategy

1. Load pretrained weights into HybridGRUFC
2. Re-initialize embedding layers (Driver/Race/Compound/Year) — real F1 categories don't match competition
3. Fine-tune ALL layers with LR = 1e-4 (0.2× pretrain LR) for 20 epochs
4. Early stopping patience=10 as normal
5. Standard 5-fold GroupKFold by Race+Year

## Inputs / Outputs
**Inputs:** `train_features_v2.parquet`, `test_features_v2.parquet`, `fold_assignments.parquet`,
`hybrid_pretrained.pt`, `hybrid_pretrain_meta.pkl`,
`oof_predictions_nb21.parquet` (for ρ comparison)

**Outputs:** `oof_predictions_nb26.parquet` (hybrid_pretrain_oof), `test_predictions_nb26.parquet`,
`models/hybrid_pretrain_fold{0-4}.pt`, `models/nb26_summary.pkl`

In [ ]:
# nb26-01  Imports
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr, rankdata
import pickle, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# nb26-02  Path detection
if Path('/kaggle/input').exists():
    candidates = list(Path('/kaggle/input').rglob('train_features_v2.parquet'))
    if not candidates:
        raise FileNotFoundError('Upload processed parquets as a Kaggle dataset first.')
    processed_dir = candidates[0].parent
    # Find pretrained weights dataset
    weight_candidates = list(Path('/kaggle/input').rglob('hybrid_pretrained.pt'))
    if not weight_candidates:
        raise FileNotFoundError(
            'Upload hybrid_pretrained.pt (from scripts/pretrain_hybrid.py) as a Kaggle dataset.')
    pretrain_dir = weight_candidates[0].parent
    models_dir   = Path('/kaggle/working/models')
    project_root = Path('/kaggle/working')
else:
    cwd = Path.cwd()
    while cwd.name != 'predict-f1-pit-stops' and cwd.parent != cwd:
        cwd = cwd.parent
    project_root  = cwd
    processed_dir = project_root / 'data' / 'processed'
    pretrain_dir  = project_root / 'models'
    models_dir    = project_root / 'models'

models_dir.mkdir(exist_ok=True)
print(f'processed_dir: {processed_dir}')
print(f'pretrain_dir : {pretrain_dir}')

train    = pd.read_parquet(processed_dir / 'train_features_v2.parquet')
test     = pd.read_parquet(processed_dir / 'test_features_v2.parquet')
folds_df = pd.read_parquet(processed_dir / 'fold_assignments.parquet')
hybrid_oof_nb21 = pd.read_parquet(processed_dir / 'oof_predictions_nb21.parquet')[['id', 'hybrid_oof']]

print(f'Train: {train.shape}, Test: {test.shape}')

In [ ]:
# nb26-03  Load pretrained weights and meta
pretrained_weights_path = pretrain_dir / 'hybrid_pretrained.pt'
pretrain_meta_path      = pretrain_dir / 'hybrid_pretrain_meta.pkl'

pretrained_state = torch.load(pretrained_weights_path, map_location='cpu')
print(f'Loaded pretrained weights: {pretrained_weights_path}')
print(f'Pretrained weight keys: {list(pretrained_state.keys())[:5]}...')

with open(pretrain_meta_path, 'rb') as f:
    pretrain_meta = pickle.load(f)
print(f'Pretrain val AUC (real F1 data): {pretrain_meta["final_val_auc"]:.4f}')

In [ ]:
# nb26-04  Settings
SEQ_COLS = [
    'LapTime (s)', 'TyreLife', 'Cumulative_Degradation_winsorized',
    'LapTime_Delta', 'Position',
]
STRAT_COLS = [
    'Stint', 'RaceProgress', 'laps_remaining', 'compound_ordinal', 'is_wet_tyre',
    'prime_pit_window', 'laps_to_driver_end', 'abs_position_change',
    'pos_change_rolling_std_3', 'PitStop_lag1',
    'TyreLife_x_laps_remaining', 'Degradation_x_RaceProgress', 'Position_x_RaceProgress',
]
assert len(STRAT_COLS) == 13

CAT_COLS       = ['Driver', 'Race', 'Compound', 'Year']
WINDOW         = 10
COMP_POS_RATE  = 0.199   # competition positive rate (~19.9%) — for bias reset + pos_weight
POS_WEIGHT     = (1 - COMP_POS_RATE) / COMP_POS_RATE  # ~4.03
BATCH_SIZE     = 2048
MAX_EPOCHS     = 50       # extended from 30 — pretrained bias correction takes ~5-10 eps
PATIENCE       = 10
FINETUNE_LR    = 1e-4     # 0.2x pretrain LR (5e-4)
WEIGHT_DECAY   = 1e-4
N_FOLDS        = 5

print(f'Fine-tune LR: {FINETUNE_LR}, max epochs: {MAX_EPOCHS}, pos_weight: {POS_WEIGHT:.2f}')
print(f'Competition positive rate: {COMP_POS_RATE:.1%}')
if DEVICE.type == 'cpu':
    print('WARNING: CPU-only. Upload to Kaggle T4.')

In [ ]:
# nb26-05  Label encoders for competition categories (fit on combined train+test)
# NOTE: These differ from the pretrain label encoders — real F1 had different drivers/races.
# The embedding layers will be re-initialized from scratch after loading pretrained weights.
combined_cats = pd.concat([train[CAT_COLS], test[CAT_COLS]], ignore_index=True)
label_encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    le.fit(combined_cats[col].astype(str))
    label_encoders[col] = le
    print(f'  {col}: {len(le.classes_)} classes')

for col in CAT_COLS:
    le = label_encoders[col]
    train[col + '_idx'] = le.transform(train[col].astype(str))
    test[col  + '_idx'] = le.transform(test[col].astype(str))

# Competition embedding dims (may differ from pretrain)
EMB_DIMS = {
    'Driver':   (len(label_encoders['Driver'].classes_)   + 1, 32),
    'Race':     (len(label_encoders['Race'].classes_)     + 1,  8),
    'Compound': (len(label_encoders['Compound'].classes_) + 1,  3),
    'Year':     (len(label_encoders['Year'].classes_)     + 1,  2),
}
print('Competition EMB_DIMS:', EMB_DIMS)

In [ ]:
# nb26-06  Build 10-lap windows  (identical to NB21)
idx_cols     = ['id', 'Race', 'Year', 'Driver', 'LapNumber']
cat_idx_cols = [c + '_idx' for c in CAT_COLS]

seq_scaler = StandardScaler()
seq_scaler.fit(train[SEQ_COLS].values)

all_sel = list(dict.fromkeys(idx_cols + SEQ_COLS + STRAT_COLS + cat_idx_cols))

combined_df = pd.concat([
    train[all_sel + ['PitNextLap']].assign(is_train=True),
    test[ all_sel].assign(is_train=False, PitNextLap=-1),
], ignore_index=True)

combined_df = combined_df.sort_values(['Race', 'Year', 'Driver', 'LapNumber']).reset_index(drop=True)
combined_df[SEQ_COLS] = seq_scaler.transform(combined_df[SEQ_COLS].values).astype(np.float32)

N        = len(combined_df)
N_SF     = len(SEQ_COLS)
seq_vals = combined_df[SEQ_COLS].values.astype(np.float32)

all_windows = np.zeros((N, WINDOW, N_SF), dtype=np.float32)
all_masks   = np.zeros((N, WINDOW),       dtype=bool)

GROUP_COLS = ['Race', 'Year', 'Driver']
print(f'Building windows for {N:,} rows...')
t0 = time.time()
for _, grp_idx in tqdm(combined_df.groupby(GROUP_COLS, sort=False).groups.items(),
                       total=combined_df[GROUP_COLS].drop_duplicates().shape[0]):
    idxs  = grp_idx.values
    n_grp = len(idxs)
    for i in range(n_grp):
        hist_len = min(i, WINDOW)
        if hist_len > 0:
            all_windows[idxs[i], WINDOW - hist_len:] = seq_vals[idxs[i - hist_len : i]]
            all_masks[idxs[i],   WINDOW - hist_len:] = True
print(f'Done in {time.time()-t0:.1f}s')

tr_mask = combined_df['is_train'].values
te_mask = ~tr_mask

train_windows   = all_windows[tr_mask]
train_seq_masks = all_masks[tr_mask]
test_windows    = all_windows[te_mask]
test_seq_masks  = all_masks[te_mask]
train_strat_raw = combined_df.loc[tr_mask, STRAT_COLS].values.astype(np.float32)
test_strat_raw  = combined_df.loc[te_mask, STRAT_COLS].values.astype(np.float32)
train_targets   = combined_df.loc[tr_mask, 'PitNextLap'].values.astype(np.float32)
train_ids       = combined_df.loc[tr_mask, 'id'].values
test_ids        = combined_df.loc[te_mask, 'id'].values
fold_lookup     = folds_df.set_index('id')['fold']
train_folds     = fold_lookup.loc[train_ids].values
train_cat       = {c: combined_df.loc[tr_mask, c+'_idx'].values for c in CAT_COLS}
test_cat        = {c: combined_df.loc[te_mask, c+'_idx'].values for c in CAT_COLS}
del all_windows, all_masks, combined_df
print(f'Train: {train_windows.shape}, Test: {test_windows.shape}')

In [ ]:
# nb26-07  Model (identical to NB21 HybridGRUFC)
import math

class HybridGRUFC(nn.Module):
    def __init__(self, n_strat=13, n_seq=5, gru_hidden=128, n_layers=2,
                 gru_drop=0.15, emb_dims=None):
        super().__init__()
        if emb_dims is None:
            emb_dims = EMB_DIMS
        self.driver_emb   = nn.Embedding(*emb_dims['Driver'])
        self.race_emb     = nn.Embedding(*emb_dims['Race'])
        self.compound_emb = nn.Embedding(*emb_dims['Compound'])
        self.year_emb     = nn.Embedding(*emb_dims['Year'])
        emb_total = sum(v[1] for v in emb_dims.values())

        self.gru = nn.GRU(n_seq, gru_hidden, n_layers, batch_first=True,
                          dropout=gru_drop if n_layers > 1 else 0.0)
        self.strat_fc = nn.Sequential(
            nn.Linear(n_strat, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 64), nn.BatchNorm1d(64), nn.ReLU())
        merge_dim = gru_hidden + 64 + emb_total
        self.head = nn.Sequential(
            nn.Linear(merge_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.25),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 1))

    def forward(self, window, mask, strat, driver, race, compound, year):
        emb = torch.cat([self.driver_emb(driver), self.race_emb(race),
                         self.compound_emb(compound), self.year_emb(year)], dim=1)
        seq_len  = mask.sum(dim=1).long().clamp(min=1)
        packed   = nn.utils.rnn.pack_padded_sequence(
            window, seq_len.cpu(), batch_first=True, enforce_sorted=False)
        _, h_n   = self.gru(packed)
        gru_feat = h_n[-1]
        strat_f  = self.strat_fc(strat)
        return self.head(torch.cat([gru_feat, strat_f, emb], dim=1)).squeeze(1)


def load_pretrained_model(pretrained_state, emb_dims):
    """Load pretrained weights, re-initialize embedding layers for competition categories.
    Also resets the output bias to match the competition's positive rate so fine-tuning
    doesn't waste epochs correcting the real-F1 bias (~3% pit rate → -3.4 log-odds)
    to the competition rate (~20% pit rate → -1.4 log-odds).
    """
    model = HybridGRUFC(emb_dims=emb_dims)
    competition_state = model.state_dict()

    emb_keys = {'driver_emb.weight', 'race_emb.weight', 'compound_emb.weight', 'year_emb.weight'}
    loaded, skipped = 0, 0
    for k, v in pretrained_state.items():
        if k in competition_state:
            if k in emb_keys:
                skipped += 1
            elif competition_state[k].shape == v.shape:
                competition_state[k] = v
                loaded += 1
            else:
                print(f'  Shape mismatch: {k} pretrain={v.shape} competition={competition_state[k].shape}')
                skipped += 1
        else:
            skipped += 1

    model.load_state_dict(competition_state)

    # Reset output bias from real-F1 log-odds to competition log-odds.
    # Pretrain: pit_rate=3.12% → bias ≈ log(0.031/0.969) ≈ -3.4
    # Competition: pit_rate=19.9% → bias ≈ log(0.199/0.801) ≈ -1.4
    # Without this, fine-tuning spends its first ~10 epochs just correcting the bias.
    with torch.no_grad():
        comp_log_odds = math.log(COMP_POS_RATE / (1 - COMP_POS_RATE))
        model.head[-1].bias.data.fill_(comp_log_odds)

    print(f'  Weights loaded: {loaded}, skipped (embeddings + shape mismatch): {skipped}')
    print(f'  Output bias reset to {comp_log_odds:.3f} (competition log-odds, pos_rate={COMP_POS_RATE:.1%})')
    return model


_m = load_pretrained_model(pretrained_state, EMB_DIMS)
print(f'Model params: {sum(p.numel() for p in _m.parameters()):,}')
del _m

In [ ]:
# nb26-08  Dataset + DataLoader  (identical to NB21)
class F1HybridDataset(Dataset):
    def __init__(self, windows, masks, strat, cat_idxs, targets=None):
        self.windows  = torch.tensor(windows, dtype=torch.float32)
        self.masks    = torch.tensor(masks,   dtype=torch.bool)
        self.strat    = torch.tensor(strat,   dtype=torch.float32)
        self.driver   = torch.tensor(cat_idxs['Driver'],   dtype=torch.long)
        self.race     = torch.tensor(cat_idxs['Race'],     dtype=torch.long)
        self.compound = torch.tensor(cat_idxs['Compound'], dtype=torch.long)
        self.year     = torch.tensor(cat_idxs['Year'],     dtype=torch.long)
        self.targets  = (None if targets is None
                         else torch.tensor(targets, dtype=torch.float32))

    def __len__(self):  return len(self.windows)

    def __getitem__(self, i):
        d = dict(window=self.windows[i], mask=self.masks[i], strat=self.strat[i],
                 driver=self.driver[i], race=self.race[i],
                 compound=self.compound[i], year=self.year[i])
        if self.targets is not None:
            d['target'] = self.targets[i]
        return d


def make_loader(windows, masks, strat, cat_idxs, targets=None, shuffle=False):
    ds = F1HybridDataset(windows, masks, strat, cat_idxs, targets)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=0, pin_memory=(DEVICE.type == 'cuda'))

In [ ]:
# nb26-09  Training utilities  (identical to NB21)
def run_epoch(model, loader, criterion=None, optimizer=None):
    is_train = optimizer is not None and criterion is not None
    model.train(is_train)
    total_loss, all_logits = 0.0, []

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for batch in loader:
            win  = batch['window'].to(DEVICE)
            mask = batch['mask'].to(DEVICE)
            st   = batch['strat'].to(DEVICE)
            drv  = batch['driver'].to(DEVICE)
            rc   = batch['race'].to(DEVICE)
            cmp  = batch['compound'].to(DEVICE)
            yr   = batch['year'].to(DEVICE)

            logits = model(win, mask, st, drv, rc, cmp, yr)

            if is_train:
                tgt  = batch['target'].to(DEVICE)
                loss = criterion(logits, tgt)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                total_loss += loss.item() * len(win)

            all_logits.append(logits.detach().cpu().float().numpy())

    probs = torch.sigmoid(torch.tensor(np.concatenate(all_logits))).numpy()
    return probs, total_loss / max(len(loader.dataset), 1)

In [ ]:
# nb26-10  5-fold fine-tuning CV
# Each fold: load pretrained weights, re-init embeddings, fine-tune at low LR.

oof_preds        = np.zeros(len(train_ids))
test_preds_folds = []
fold_aucs        = []
best_epochs      = []

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))

for fold in range(N_FOLDS):
    print(f'\n{"="*60}\nFold {fold} (pretrained fine-tune)\n{"="*60}')

    tr_idx = np.where(train_folds != fold)[0]
    va_idx = np.where(train_folds == fold)[0]

    strat_scaler = StandardScaler()
    tr_strat = strat_scaler.fit_transform(train_strat_raw[tr_idx])
    va_strat = strat_scaler.transform(train_strat_raw[va_idx])
    te_strat = strat_scaler.transform(test_strat_raw)

    tr_cat = {c: train_cat[c][tr_idx] for c in CAT_COLS}
    va_cat = {c: train_cat[c][va_idx] for c in CAT_COLS}

    tr_loader = make_loader(train_windows[tr_idx], train_seq_masks[tr_idx],
                            tr_strat, tr_cat, targets=train_targets[tr_idx], shuffle=True)
    va_loader = make_loader(train_windows[va_idx], train_seq_masks[va_idx], va_strat, va_cat)
    te_loader = make_loader(test_windows, test_seq_masks, te_strat, test_cat)

    # Load pretrained weights for every fold (independent fine-tune per fold)
    model     = load_pretrained_model(pretrained_state, EMB_DIMS).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=FINETUNE_LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

    best_auc, best_epoch, patience_ctr, best_state = 0.0, 0, 0, None

    for epoch in range(MAX_EPOCHS):
        _, tr_loss   = run_epoch(model, tr_loader, criterion, optimizer)
        scheduler.step()
        val_probs, _ = run_epoch(model, va_loader)
        val_auc      = roc_auc_score(train_targets[va_idx], val_probs)

        if val_auc > best_auc:
            best_auc, best_epoch, patience_ctr = val_auc, epoch, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1

        print(f'  ep {epoch+1:3d}  loss={tr_loss:.4f}  val={val_auc:.4f}  '
              f'best={best_auc:.4f} (ep{best_epoch+1})')

        if patience_ctr >= PATIENCE:
            print('  Early stop')
            break

    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    oof_probs, _ = run_epoch(model, va_loader)
    oof_preds[va_idx] = oof_probs
    fold_auc = roc_auc_score(train_targets[va_idx], oof_probs)
    fold_aucs.append(fold_auc)
    best_epochs.append(best_epoch + 1)
    print(f'  Fold {fold} AUC: {fold_auc:.4f}  (best ep {best_epoch+1})')

    te_probs, _ = run_epoch(model, te_loader)
    test_preds_folds.append(te_probs)

    torch.save(best_state, models_dir / f'hybrid_pretrain_fold{fold}.pt')

oof_auc = roc_auc_score(train_targets, oof_preds)
print(f'\nOOF AUC : {oof_auc:.4f}')
print(f'Fold AUCs: {[f"{a:.4f}" for a in fold_aucs]}')
print(f'Fold std : {np.std(fold_aucs):.4f}')
print(f'Best eps : {best_epochs}')

In [ ]:
# nb26-11  Compare to NB21 baseline + correlation analysis
nb21_vals  = hybrid_oof_nb21.set_index('id').loc[train_ids, 'hybrid_oof'].values
lgbm_df    = pd.read_parquet(processed_dir / 'oof_predictions_nb12.parquet')
lgbm_vals  = lgbm_df.set_index('id').loc[train_ids, 'lgbm_tuned_oof'].values

rho_nb21, _ = spearmanr(oof_preds, nb21_vals)
rho_lgbm, _ = spearmanr(oof_preds, lgbm_vals)

nb21_auc = roc_auc_score(train_targets, nb21_vals)
lgbm_auc = roc_auc_score(train_targets, lgbm_vals)

print(f'NB26 Pretrained solo OOF AUC : {oof_auc:.4f}')
print(f'NB21 Baseline  solo OOF AUC  : {nb21_auc:.4f}  (no pretraining)')
print(f'LGBM NB12      solo OOF AUC  : {lgbm_auc:.4f}')
print(f'Delta vs NB21                : {oof_auc - nb21_auc:+.4f}')
print()
print('Spearman rho:')
print(f'  NB26 x NB21 : {rho_nb21:.4f}  (same arch — expect 0.93+; pretraining adds noise)')
print(f'  NB26 x LGBM : {rho_lgbm:.4f}')
print()

# Rank-avg preview
def rank_norm(a): return rankdata(a) / len(a)
ens_pretrain_nb21 = roc_auc_score(train_targets,
    (rank_norm(oof_preds) + rank_norm(nb21_vals)) / 2)
ens_pretrain_lgbm = roc_auc_score(train_targets,
    (rank_norm(oof_preds) + rank_norm(lgbm_vals)) / 2)

print(f'Rank avg NB26+NB21 : {ens_pretrain_nb21:.4f}  (delta vs NB21: {ens_pretrain_nb21 - nb21_auc:+.4f})')
print(f'Rank avg NB26+LGBM : {ens_pretrain_lgbm:.4f}')

# Decision
print()
if oof_auc > nb21_auc + 0.002:
    print(f'DECISION: Pretraining improved by +{oof_auc - nb21_auc:.4f}. Submit NB26 solo.')
    print(f'  Proj LB (solo +0.0177 boost): {oof_auc + 0.0177:.4f}')
elif oof_auc > nb21_auc:
    print(f'DECISION: Marginal improvement (+{oof_auc - nb21_auc:.4f}). Consider multi-seed NB26.')
else:
    print(f'DECISION: Pretraining did NOT help (Δ={oof_auc - nb21_auc:+.4f}).')
    print('  Root cause: feature distribution mismatch real→synthetic is too large.')
    print('  Recommendation: try fixing seq/strat scaler alignment before fine-tuning.')

In [ ]:
# nb26-12  Average test predictions + save outputs
test_preds_mean = np.mean(np.stack(test_preds_folds, axis=0), axis=0)

oof_df = pd.DataFrame({
    'id':                  train_ids,
    'fold':                train_folds,
    'PitNextLap':          train_targets.astype(int),
    'hybrid_pretrain_oof': oof_preds,
})
assert len(oof_df) == 439140
oof_df.to_parquet(processed_dir / 'oof_predictions_nb26.parquet', index=False)
print('Saved oof_predictions_nb26.parquet')

test_out = pd.DataFrame({'id': test_ids, 'hybrid_pretrain_pred': test_preds_mean})
test_out.to_parquet(processed_dir / 'test_predictions_nb26.parquet', index=False)
print('Saved test_predictions_nb26.parquet')

summary = {
    'oof_auc':           oof_auc,
    'fold_aucs':         fold_aucs,
    'fold_std':          float(np.std(fold_aucs)),
    'best_epochs':       best_epochs,
    'delta_vs_nb21':     float(oof_auc - nb21_auc),
    'rho_vs_nb21':       float(rho_nb21),
    'rho_vs_lgbm':       float(rho_lgbm),
    'pretrain_val_auc':  pretrain_meta.get('final_val_auc', None),
    'model':             'HybridGRUFC-NB26-Pretrained',
    'finetune_lr':       FINETUNE_LR,
    'max_epochs':        MAX_EPOCHS,
}
with open(models_dir / 'nb26_summary.pkl', 'wb') as f:
    pickle.dump(summary, f)
print('Saved nb26_summary.pkl')

print(f'\n=== NB26 Summary ===')
for k, v in summary.items():
    print(f'  {k}: {v}')

In [ ]:
# nb26-13  Submission (if NB26 beats NB21 meaningfully)
SUBMISSION_DIR = project_root / 'submissions'
SUBMISSION_DIR.mkdir(exist_ok=True)

if oof_auc > nb21_auc + 0.002:
    sub = pd.DataFrame({'id': test_ids, 'PitNextLap': test_preds_mean})
    sub_path = SUBMISSION_DIR / f'submission_v014_pretrained_hybrid_cv{oof_auc:.4f}.csv'
    sub.to_csv(sub_path, index=False)
    print(f'Submission saved: {sub_path}')
    print(f'CV OOF AUC: {oof_auc:.4f}')
    print(f'Projected LB (solo +0.0177 boost): {oof_auc + 0.0177:.4f}')
else:
    print(f'NB26 AUC ({oof_auc:.4f}) not better enough than NB21 ({nb21_auc:.4f}).')
    print('Do not submit NB26. Pretraining gain is within noise floor.')